In [1]:
!pip install boto3 tqdm pillow

import boto3
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from io import BytesIO
from tqdm import tqdm

# --- R2 BİLGİLERİ ---
ACCESS_KEY = '26ff1aa1645b69cac212507ec4c079dd'
SECRET_KEY = '37872cda19b45e33509d5b84a3514234a5a06755481363e1561cb7a404f23a77'
ENDPOINT_URL = 'https://8a97a7792885ace66acb04fa79b9ded7.r2.cloudflarestorage.com' # Örn: https://<hesap-id>.r2.cloudflarestorage.com
BUCKET_NAME = 'genrevision'

# Boto3 istemcisi (R2'dan resim çekmek için)
s3_client = boto3.client(
    service_name='s3',
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name='auto'
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.9 MB/s eta 0:00:00


In [4]:
# --- VERİ SETİ VE DATALOADER HAZIRLIĞI ---
class R2MovieDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
        self.label_cols = [col for col in self.df.columns if col.startswith('Tur_')]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        film_id = self.df.iloc[idx]['id']
        file_key = f"{film_id}.jpg"

        try:
            # Resmi doğrudan R2'dan çekiyoruz
            response = s3_client.get_object(Bucket=BUCKET_NAME, Key=file_key)
            image = Image.open(BytesIO(response['Body'].read())).convert('RGB')
        except Exception as e:
            # Resim çekilemezse siyah kare döndür (eğitimin çökmesini engeller)
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(self.df.iloc[idx][self.label_cols].values.astype('float32'))
        return image, labels

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = R2MovieDataset(csv_file='yapay_zeka_hazir_veri.csv', transform=transform)

# İŞTE EKSİK OLAN DATALOADER BURADA TANIMLANIYOR
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
num_classes = len(dataset.label_cols)
print(f"Tespit edilecek tür sayısı: {num_classes}")

# --- MODEL VE EĞİTİM DÖNGÜSÜ ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15

print("\n--- EĞİTİM BAŞLIYOR ---")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    # tqdm döngüsü şimdi dataloader'ı bulabilecek
    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1} Bitti! Ortalama Loss: {epoch_loss:.4f}")

torch.save(model.state_dict(), "genre_vision_final.pth")
print("\nModel eğitildi ve 'genre_vision_final.pth' adıyla kaydedildi!")

Tespit edilecek tür sayısı: 19
Kullanılan cihaz: cuda

--- EĞİTİM BAŞLIYOR ---


Epoch 1/15: 100%|██████████| 28/28 [07:22<00:00, 15.80s/it, loss=0.176]


Epoch 1 Bitti! Ortalama Loss: 0.2546


Epoch 2/15: 100%|██████████| 28/28 [03:22<00:00,  7.22s/it, loss=0.192]


Epoch 2 Bitti! Ortalama Loss: 0.1786


...

Epoch 13 Bitti! Ortalama Loss: 0.0021


Epoch 14/15: 100%|██████████| 28/28 [03:15<00:00,  6.99s/it, loss=0.000927]


Epoch 14 Bitti! Ortalama Loss: 0.0013


Epoch 15/15: 100%|██████████| 28/28 [03:05<00:00,  6.63s/it, loss=0.00116]

Epoch 15 Bitti! Ortalama Loss: 0.0010

Model eğitildi ve 'genre_vision_final.pth' adıyla kaydedildi!


In [5]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import requests
from io import BytesIO

# --- 1. MODELİ TEST İÇİN HAZIRLAMA ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CSV'deki kolon isimlerinden aldığımız 19 tür (Sıralama CSV ile birebir aynı olmalı)
turler = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery',
    'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western'
]

# Modeli tekrar kuruyoruz
num_classes = len(turler)
model = models.resnet18(weights=None) # Ağırlıkları baştan indirmemize gerek yok
model.fc = nn.Linear(model.fc.in_features, num_classes)

# Eğittiğimiz ağırlıkları (.pth) modele yüklüyoruz
model.load_state_dict(torch.load('genre_vision_final.pth', map_location=device))
model = model.to(device)
model.eval() # Modeli "değerlendirme/tahmin" moduna alıyoruz

# --- 2. GÖRSEL İŞLEME VE TAHMİN FONKSİYONU ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def tur_tahmin_et(url, baraj=0.3): # %30'dan yüksek ihtimalli türleri göstersin
    print(f"Görsel analiz ediliyor...\n")

    # URL'den resmi çek
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')

    # Modeli için tensöre çevir ve boyutu ayarla
    img_tensor = transform(img).unsqueeze(0).to(device)

    # Tahmin yap
    with torch.no_grad(): # Tahmin yaparken gradyan hesaplamaya gerek yok (bellek tasarrufu)
        raw_outputs = model(img_tensor)

        # Çıktıları Sigmoid fonksiyonundan geçirerek 0-1 arası (0-100%) yüzdelere çeviriyoruz
        probabilities = torch.sigmoid(raw_outputs).squeeze()

    # Sonuçları ekrana yazdır
    print("--- MODELİN TAHMİNLERİ ---")
    for i, prob in enumerate(probabilities):
        yuzde = prob.item() * 100
        if yuzde >= (baraj * 100):
            print(f"✅ {turler[i]}: %{yuzde:.2f}")
        elif yuzde > 10: # %10 ile %30 arasında kalan zayıf ihtimaller
            print(f"❔ {turler[i]}: %{yuzde:.2f}")

# --- 3. TEST ZAMANI! ---
# İnternetten herhangi bir film afişi URL'si koyabilirsin (Örn: The Matrix)
test_url = "https://a.ltrbxd.com/resized/film-poster/4/0/3/8/3/4/403834-ayla-the-daughter-of-war-0-2000-0-3000-crop.jpg?v=a3e983026a"

tur_tahmin_et(test_url)

Görsel analiz ediliyor...

--- MODELİN TAHMİNLERİ ---
✅ Drama: %53.18
